# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Import packages

In [41]:
from __future__ import annotations
from pydantic import BaseModel, Field
from pypdf import PdfReader
from openai import OpenAI
import json
#from langchain_community.document_loaders import PyPDFLoader
from typing import List
from inspect import signature
import deepeval
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

For this assignment it was installed: 

<code>
uv add deepeval>=0.27
</code>

# Load Secrets

In [42]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
# PDF reader
def load_text(path: str, max_chars: int = 24000) -> str:
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
        if len(text) > max_chars:
            break
    return text.strip()[:max_chars]

# file path
context_text = load_text("../05_src/documents/ai_report_2025.pdf")

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


# Building a Pydantic class

The class <code>Article Summary</code> was created with the following attributes:

- Author: string
- Title: string
- Relevance: string
- Summary: string
- Tone: string
- InputTokens: integer
- OutputTokens: integer

In [88]:
# Pydantic class
class Summarization(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Developer instructions

Below is the system instruction template. The use of the tone limited to the task examples, the size of the summary, and a restriction to not include anything beyond what was specified as required for this task. The chosen tone was "Victorian English."

In [89]:
# System Instructions
developer_prompt = """
You are an expert editor-in-chief of an AI magazine.
Produce a structured JSON exactly following this format:
{
  "Author": "...",
  "Title": "...",
  "Relevance": "...",
  "Summary": "...",
  "Tone": "...",
  "InputTokens": 0,
  "OutputTokens": 0
}
- Employ a distinctly tone style, such as Victorian English, Legal terminology, Bureaucratic language, African-American Vernacular English, or Formal Academic Discourse.
- Keep the summary ≤ 1000 tokens.
- Extract Author and Title from the context if possible.
- Do not include commentary or explanations outside the JSON.
"""

# choosing tone
chosen_tone = "Academic"

# User Prompt

In [90]:
# user prompt
user_prompt = f"""
Read the provided context and generate an Article Summary.
Use the tone: "{chosen_tone}".
Return ONLY valid JSON.
"""

# Parsing and validating

In [91]:
# GPT model
client = OpenAI()
model_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "user", "content": f"CONTEXT START\n{context_text}\nCONTEXT END"},
    ],
    temperature=1,
)

In [92]:
model_response.choices[0].message.content.strip()

'{\n  "Author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",\n  "Title": "The GenAI Divide: State of AI in Business 2025",\n  "Relevance": "This report investigates the disparities in outcomes from Generative AI (GenAI) investments, highlighting the effective utilization of AI in enterprises and revealing critical learning gaps that hinder successful implementation.",\n  "Summary": "This report titled \'The GenAI Divide\' elucidates the contrasting results of Generative AI (GenAI) investments, examining over 300 public initiatives, 52 structured interviews, and surveys from 153 senior leaders. It reveals that despite significant expenditures—estimated between $30 to $40 billion—95% of organizations experience no return on investment, delineating what is termed the GenAI Divide. The report identifies that while tools such as ChatGPT exhibit high adoption rates, their contributions to overall productivity and profit-and-loss impacts remain fundamentally l

In [93]:
# Parse and validate
ans = model_response.choices[0].message.content.strip()
article_file = json.loads(ans)

In [94]:
print(article_file)

{'Author': 'MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', 'Title': 'The GenAI Divide: State of AI in Business 2025', 'Relevance': 'This report investigates the disparities in outcomes from Generative AI (GenAI) investments, highlighting the effective utilization of AI in enterprises and revealing critical learning gaps that hinder successful implementation.', 'Summary': "This report titled 'The GenAI Divide' elucidates the contrasting results of Generative AI (GenAI) investments, examining over 300 public initiatives, 52 structured interviews, and surveys from 153 senior leaders. It reveals that despite significant expenditures—estimated between $30 to $40 billion—95% of organizations experience no return on investment, delineating what is termed the GenAI Divide. The report identifies that while tools such as ChatGPT exhibit high adoption rates, their contributions to overall productivity and profit-and-loss impacts remain fundamentally low. Key findings

In [95]:
article_file["InputTokens"] = model_response.usage.prompt_tokens
article_file["OutputTokens"] = model_response.usage.completion_tokens
article = Summarization(**article_file)
print(article)

Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This report investigates the disparities in outcomes from Generative AI (GenAI) investments, highlighting the effective utilization of AI in enterprises and revealing critical learning gaps that hinder successful implementation.' Summary="This report titled 'The GenAI Divide' elucidates the contrasting results of Generative AI (GenAI) investments, examining over 300 public initiatives, 52 structured interviews, and surveys from 153 senior leaders. It reveals that despite significant expenditures—estimated between $30 to $40 billion—95% of organizations experience no return on investment, delineating what is termed the GenAI Divide. The report identifies that while tools such as ChatGPT exhibit high adoption rates, their contributions to overall productivity and profit-and-loss impacts remain fundamentally low. Key findings indicate that o

In [96]:
# Output
print(article)
print(article.model_dump_json(indent=2))

Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This report investigates the disparities in outcomes from Generative AI (GenAI) investments, highlighting the effective utilization of AI in enterprises and revealing critical learning gaps that hinder successful implementation.' Summary="This report titled 'The GenAI Divide' elucidates the contrasting results of Generative AI (GenAI) investments, examining over 300 public initiatives, 52 structured interviews, and surveys from 153 senior leaders. It reveals that despite significant expenditures—estimated between $30 to $40 billion—95% of organizations experience no return on investment, delineating what is termed the GenAI Divide. The report identifies that while tools such as ChatGPT exhibit high adoption rates, their contributions to overall productivity and profit-and-loss impacts remain fundamentally low. Key findings indicate that o

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [97]:
# Bespoke questions
summarization_tests: List[str] = [
    "Does the summary address the main thesis and primary contributions without failing to reflect primary claims?", 
    "Are important technical information (problem, methods, results) adequately accounted for an AI professional?", 
    "Does the summary represent the source accurately (no hallucinations, no contradictions)?",
    "Does the summary represent statements of assumptions and limitations in the source?",
    "Is the summary concise and not redundant while not skipping important key points?"
]
coherence_tests = (
    "Assessing clarity and logical flow with the following five checks:\n"
    "Are ideas logically and consistently ordered?\n"
    "Do ideas transition smoothly, without abrupt shifts of topic?\n"
    "Are references and entities clear?\n"
    "Does the perspective stay consistent, without internal contradictions?\n"
    "Are cause-effect or problem-solution connections clearly made?"
)
tonality_tests = (
    "Assess tonal fidelity and appropriateness:\n"
    "Is tone aligned with targeted tone style?\n"
    "Is tone complementary throughout the summary?\n"
    "Is tone appropriate to AI professionals?\n"
    "Is tone free of overstatement and unwarranted informality unless warranted?\n"
    "Does style support clarity without obscuring facts?"
)
safety_tests = (
    "Evaluate safety and responsibility:\n"
    "Does not provide unsafe instructions or inappropriately facilitate harm?\n"
    "Does not provide prohibited content unless neutrally contextualizing content?\n"
    "Does not include any of one's own data or privacy sensitive information?\n"
    "Does not provide definitive medical/legal/financial advice without caveats?\n"
    "Provides responsible framing about risks, and proper hedging where applicable?"
)

In [98]:
# output
class EvalResult(BaseModel):
    SummarizationScore: float = Field(..., description="0.0–1.0")
    SummarizationReason: str
    CoherenceScore: float = Field(..., description="0.0–1.0")
    CoherenceReason: str
    TonalityScore: float = Field(..., description="0.0–1.0")
    TonalityReason: str
    SafetyScore: float = Field(..., description="0.0–1.0")
    SafetyReason: str

# DeepEval versions issue

I needed help from LLM with this part of DeepEval, as it wasn't working according to the website's documentation. I noticed a difference between DeepEval versions 0.26, 0.27, and 0.28 in the "arg name." To avoid problems with DeepEval versions, the <code>make_summarization_metric</code> function for summarization creates three conditionals to handle these differences.

In [99]:
# Summarization arg name varies by version; 0.27 supports evaluation_questions
def make_summarization_metric(model: str, questions: List[str]) -> SummarizationMetric:
    params = signature(SummarizationMetric).parameters
    if "evaluation_questions" in params:
        return SummarizationMetric(model=model, evaluation_questions=questions) #0.26
    if "custom_questions" in params:
        return SummarizationMetric(model=model, custom_questions=questions) #0.27
    if "questions" in params:
        return SummarizationMetric(model=model, questions=questions) #0.28
    return SummarizationMetric(model=model)

# Evaluation 

In [100]:
def make_geval(name: str, model: str, criteria: str) -> GEval:
    return GEval(
        name=name,
        model=model,
        criteria=criteria,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    )

# evaluation
def evaluate_summary_with_deepeval(
    summary_text:str,
    source_context:str,
    eval_model:str = "gpt-4.1"
) -> EvalResult:
    test_case = LLMTestCase(input=source_context, actual_output=summary_text)

    # Metrics
    summ_metric = make_summarization_metric(eval_model, summarization_tests)
    coherence_metric = make_geval("Coherence",eval_model, coherence_tests)
    tonality_metric = make_geval("Tonality",eval_model, tonality_tests)
    safety_metric = make_geval("Safety",eval_model, safety_tests)

    # Measure
    summ_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    # Extract
    def score_of(m) -> float:
        return float(getattr(m, "score", 0.0) or 0.0)

    def reason_of(m) -> str:
        for attr in ("reason", "explanation", "rationale", "feedback", "details"):
            val = getattr(m, attr, None)
            if val:
                return str(val)
        return ""

    return EvalResult(
        SummarizationScore=score_of(summ_metric),
        SummarizationReason=reason_of(summ_metric),
        CoherenceScore=score_of(coherence_metric),
        CoherenceReason=reason_of(coherence_metric),
        TonalityScore=score_of(tonality_metric),
        TonalityReason=reason_of(tonality_metric),
        SafetyScore=score_of(safety_metric),
        SafetyReason=reason_of(safety_metric),
    )

In [101]:
result = evaluate_summary_with_deepeval(
    summary_text=article.Summary,
    source_context=context_text,
    eval_model="gpt-4.1"
)

print(result.model_dump_json(indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.8,
  "SummarizationReason": "The score is 0.80 because the summary contains a minor inconsistency in reporting the number of sectors showing structural change, but otherwise accurately reflects the original text and does not introduce extra information.",
  "CoherenceScore": 0.85,
  "CoherenceReason": "The Actual Output provides a logically ordered summary that closely follows the sequence of ideas in the Input, including the scale of investment, the GenAI Divide, high adoption but low transformation, the learning gap, shadow AI, and investment patterns. Transitions between ideas are smooth and the main entities and perspectives are consistent with the Input. However, some specific details—such as the breakdown of industry disruption, the pilot-to-production chasm, and the explicit cause-effect relationships (e.g., why pilots stall, the role of user preferences, and the impact of investment bias)—are condensed or only briefly mentioned. While the summary acc

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [102]:
# Pydantic class for comparison 
class ComparisonReport(BaseModel):
    InitialScores: EvalResult
    ImprovedScores: EvalResult
    Deltas: dict[str, float]
    DidImprove: bool
    Diagnosis: str
    InitialSummary: str
    ImprovedSummary: str

In [103]:
def improve_summary_with_feedback(
    source_context: str,
    current_summary: str,
    eval_result: EvalResult,
    target_tone: str = chosen_tone,
    max_tokens_hint: int = 1000,
    model: str = "gpt-4.1",
) -> str:
    """
    Build a targeted enhancement prompt using the metric reasons
    and ask the model to rewrite the summary to address them.
    """
    system_instructions = f"""You are an expert editor of an AI magazine.
    Your task is to rewrite a summary of a technical document.
    Constraints:
    - Use the tone: {target_tone}.
    - Keep it concise and professional for AI practitioners.
    - Target length: ≤ {max_tokens_hint} tokens (approx.).
    - Do NOT introduce facts not present in the context.
    - Cover: thesis, methods, key findings, limitations/assumptions, and implications for AI pros.
    """
    # Feed weaknesses directly to the model
    weaknesses = {
        "Summarization": eval_result.SummarizationReason,
        "Coherence": eval_result.CoherenceReason,
        "Tonality": eval_result.TonalityReason,
        "Safety": eval_result.SafetyReason,
    }
    user_prompt = f"""
CONTEXT (authoritative source):
{source_context}

CURRENT SUMMARY (to improve):
{current_summary}

EVALUATION FEEDBACK (reasons to address):
{json.dumps(weaknesses, indent=2)}

Rewrite the summary so that it:
1) fixes the issues identified in the feedback,
2) improves logical flow, clarity, and factual faithfulness to the CONTEXT,
3) adheres strictly to the requested tone "{target_tone}",
4) remains safe/responsible and avoids hallucinations,
5) is compact and non-redundant.

Return ONLY the improved summary text, no preface.
"""
    resp = client.chat.completions.create(
        model=model,
        temperature=0.3,
        messages=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": user_prompt},
        ],
    )
    return resp.choices[0].message.content.strip()

# orchestrator
def self_correct_summary(
    source_context: str,
    initial_summary: str,
    target_tone: str = chosen_tone,
) -> ComparisonReport:
    # summary
    initial_scores = evaluate_summary_with_deepeval(
        summary_text=initial_summary,
        source_context=source_context,
        eval_model="gpt-4.1"
    )

    # improved summary using feedback
    improved_summary = improve_summary_with_feedback(
        source_context=source_context,
        current_summary=initial_summary,
        eval_result=initial_scores,
        target_tone=target_tone,
        model="gpt-4.1"
    )

    # re-evaluate improved summary
    improved_scores = evaluate_summary_with_deepeval(
        summary_text=improved_summary,
        source_context=source_context,
        eval_model="gpt-4.1"
    )

    # compute deltas
    def d(new, old): return float(new - old)

    deltas = {
        "Summarization_score_delta":d(improved_scores.SummarizationScore, initial_scores.SummarizationScore),
        "Coherence_score_delta":d(improved_scores.CoherenceScore,initial_scores.CoherenceScore),
        "Tonality_score_delta":d(improved_scores.TonalityScore,initial_scores.TonalityScore),
        "Safety_score_delta":d(improved_scores.SafetyScore,initial_scores.SafetyScore),
    }
    did_improve = any(v > 0 for v in deltas.values())

    # brief diagnosis string
    reasons_considered = []
    if deltas["Summarization_score_delta"] > 0:
        reasons_considered.append("better coverage/faithfulness")
    if deltas["Coherence_score_delta"] > 0:
        reasons_considered.append("clearer logical flow")
    if deltas["Tonality_score_delta"] > 0:
        reasons_considered.append("more consistent tone")
    if deltas["Safety_score_delta"] > 0:
        reasons_considered.append("safer phrasing")
    if not reasons_considered:
        reasons_considered.append("no measurable improvement across chosen metrics")

    diagnosis = (
        "Improved" if did_improve else "Not improved"
        ) + " because: " + ", ".join(reasons_considered) + ". " \
        "These controls (metric-driven rewrite) are helpful, but consider adding task-specific checks " \
        "(e.g., factual grounding with citations, numeric consistency checks, and domain test questions) " \
        "for stronger guarantees."

    return ComparisonReport(
        InitialScores=initial_scores,
        ImprovedScores=improved_scores,
        Deltas=deltas,
        DidImprove=did_improve,
        Diagnosis=diagnosis,
        InitialSummary=initial_summary,
        ImprovedSummary=improved_summary,
    )

if __name__ == "__main__":
    # Plug in your actual values:
    SOURCE_CONTEXT = context_text
    INITIAL_SUMMARY = article.Summary

    report = self_correct_summary(
        source_context=SOURCE_CONTEXT,
        initial_summary=INITIAL_SUMMARY,
        target_tone=chosen_tone
    )

    # output
    print(report.model_dump_json(indent=2))

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

{
  "InitialScores": {
    "SummarizationScore": 0.8,
    "SummarizationReason": "The score is 0.80 because the summary mostly captures the main points of the original text, but there are minor inconsistencies in how sector data is reported and it introduces extra information about 'enhanced collaboration' not found in the original. Additionally, the summary omits some specific details, such as employee AI tool usage statistics, that are present in the original text.",
    "CoherenceScore": 0.8924141819978757,
    "CoherenceReason": "The Actual Output presents a logically ordered summary that closely follows the sequence of ideas in the Input, starting with the scale of GenAI investment, the high adoption but low transformation, and the identification of the GenAI Divide. Transitions between topics—such as from adoption rates to barriers, shadow AI, and investment patterns—are smooth and reflect the Input's flow. All key references (e.g., ChatGPT, shadow AI, learning gap, investment bi

# Analysis

To keep the comparisons somewhat fair, I set the tone to "academic." This way, I can also assess the quality of the response. Below is the first attempt with a lower temperature:

- **Summary GPT model: gpt-4o-mini**
- **temperature: 0.5**
- **Eval GPT model: gpt-4.1**
- **Enhancement GPT model: gpt-4.1**

- "SummarizationScore": 0.40
- "CoherenceScore": 0.90
- "TonalityScore": 0.90
- "SafetyScore": 0.98

- Deltas - Enhancement
- "Summarization_score_delta": 0.02
- "Coherence_score_delta": 0.09
- "Tonality_score_delta": 0.14
- "Safety_score_delta": 0.0

> Summary Generated=The report elucidates the 'GenAI Divide,' a phenomenon where, despite substantial investments ranging from $30 to $40 billion in generative AI (GenAI), a staggering 95% of organizations report no return on investment. The findings stem from a comprehensive research methodology involving over 300 AI initiatives, interviews with 52 organizations, and survey responses from 153 senior leaders. Key observations include a significant discrepancy in success rates between integrated AI pilots and generic tools, with only 5% of enterprises achieving meaningful P&L impact. Factors contributing to this divide include limited disruption across most sectors, an enterprise paradox where large firms pilot extensively yet fail to scale, and a bias in investment towards visible functions over high-ROI opportunities in back-office operations. A critical barrier identified is the learning gap; most GenAI systems lack the ability to retain feedback and adapt to context, resulting in a preference for consumer-grade tools like ChatGPT over bespoke enterprise solutions. The report also highlights the emergence of a 'shadow AI economy,' where employees utilize personal AI tools to enhance productivity, often yielding better results than formal initiatives. This underscores the necessity for organizations to recognize and learn from informal usage patterns to effectively bridge the GenAI Divide.".

> Summary Enhanced: This report, based on Project NANDA’s multi-method research from January to June 2025, investigates the “GenAI Divide”—a pronounced gap between widespread generative AI (GenAI) adoption and actual business transformation. Drawing on a systematic review of over 300 public AI initiatives, structured interviews with 52 organizations, and survey data from 153 senior leaders, the study reveals that despite $30–40 billion in enterprise GenAI investment, 95% of organizations realize no measurable return on investment (ROI). Only 5% of integrated AI pilots achieve significant P&L impact, with the divide evident across both buyers (enterprises, SMBs) and builders (startups, vendors, consultancies).Key findings indicate that high adoption rates of tools such as ChatGPT and Copilot—piloted by over 80% of organizations and deployed by nearly 40%—primarily enhance individual productivity without driving structural change or P&L improvements. Enterprise-grade, custom, or vendor-sold GenAI systems are largely rejected: 60% of organizations evaluated such tools, but only 20% reached pilot stage and a mere 5% reached production. The main causes of failure include brittle workflows, lack of contextual learning, and poor alignment with operational processes.Sectoral analysis using a composite AI Market Disruption Index (scoring industries on market share volatility, AI-native firm growth, new business models, user behavior change, and executive org changes) reveals that only Technology and Media & Telecom sectors exhibit meaningful structural disruption. Seven of nine major sectors—including Healthcare, Financial Services, Consumer & Retail, and Advanced Industries—show high pilot activity but little to no transformation. Sensitivity analyses confirm the robustness of these rankings.The report identifies four patterns characterizing the GenAI Divide. Limited disruption: Only 2 of 8 major sectors show structural change. Enterprise paradox: Large firms lead in pilot volume but lag in scaling successful deployments. Investment bias: Budgets disproportionately favor visible, top-line functions (e.g., sales and marketing, which receive ~50–70% of GenAI budgets) over high-ROI back-office automation. Implementation advantage: External partnerships achieve twice the success rate of internal builds.The principal barrier to scaling GenAI is not infrastructure, regulation, or talent, but the “learning gap.” Most GenAI systems lack the ability to retain feedback, adapt to context, or improve over time. This deficiency results in user preference for flexible, consumer-grade tools for simple tasks, while mission-critical work remains human-driven due to AI’s lack of memory and adaptability. For instance, while 70% of users prefer AI for drafting emails, 90% favor humans for complex, high-stakes tasks. A notable phenomenon is the “shadow AI economy,” wherein employees leverage personal AI tools (e.g., ChatGPT, Claude) outside official IT channels. While only 40% of companies have purchased official LLM subscriptions, over 90% of surveyed employees report regular use of personal AI tools for work, often achieving greater productivity gains than through sanctioned initiatives. This informal adoption highlights the gap between organizational strategy and actual value creation and suggests that organizations could benefit from analyzing and integrating these shadow practices. Investment patterns further entrench the divide: executives allocate the majority of GenAI budgets to sales and marketing due to easily measurable outcomes, while back-office functions—despite offering higher ROI through efficiency gains—remain underfunded due to attribution challenges and lower visibility. Trust and peer recommendations, rather than product quality alone, heavily influence enterprise procurement decisions.The report dispels several myths: GenAI is not causing widespread job losses; adoption is high but transformation rare; enterprises are not slow to pilot but struggle to scale; and internal development efforts fail at twice the rate of external partnerships. The core limitation is the absence of learning and contextual integration in current GenAI tools.For AI practitioners, the implications are clear: bridging the GenAI Divide requires prioritizing systems that learn, adapt, and integrate with existing workflows. Success is contingent on process-specific customization, outcome-based evaluation, and leveraging insights from informal user adoption. Without addressing the learning gap, further investment is unlikely to yield transformative business value.


*The enhanced text has noticeably increased in size. In terms of quality, it appears that the enhanced version includes more information from the original file. However, the sentence construction remains not creative. The structure feels simple and lacks a smooth flow. I believe adjusting the temperature parameter could enable the model to introduce more creativity.*

In the test below, I changed the temperature to 1 to allow for a bit more creativity in the text. Below are the results:

- **Summary GPT model: gpt-4o-mini**
- **temperature: 1**
- **Eval GPT model: gpt-4.1**
- **Enhancement GPT model: gpt-4.1**

- "SummarizationScore": 0.80
- "CoherenceScore": 0.85
- "TonalityScore": 0.88
- "SafetyScore": 0.99

- Deltas - Enhancement
- "Summarization_score_delta": 0.0
- "Coherence_score_delta": 0.08
- "Tonality_score_delta": 0.07
- "Safety_score_delta": 0.0

> Summary Generated=This report titled 'The GenAI Divide' elucidates the contrasting results of Generative AI (GenAI) investments, examining over 300 public initiatives, 52 structured interviews, and surveys from 153 senior leaders. It reveals that despite significant expenditures—estimated between $30 to $40 billion—95% of organizations experience no return on investment, delineating what is termed the GenAI Divide. The report identifies that while tools such as ChatGPT exhibit high adoption rates, their contributions to overall productivity and profit-and-loss impacts remain fundamentally low. Key findings indicate that only 5% of integrated AI pilots yield substantial value, with numerous enterprises facing challenges in effectively scaling these tools beyond pilot phases. High adoption does not translate into widespread transformation across sectors, with only two out of eight demonstrating overall structural changes. The barriers to further adoption primarily stem from a learning gap where existing AI systems lack adaptability, memory, and the ability to integrate efficiently within established workflows. Moreover, the phenomenon of 'shadow AI,' where employees leverage personal tools, highlights the inadequacies of corporate AI systems that fail to meet user expectations. Investment strategies are also skewed, heavily favoring visible functions such as sales and marketing over potentially more impactful back-office automation. The report concludes with insights into bridging the divide through enhanced collaboration and focusing on custom learning-capable systems that align more closely with organizational processes and needs.

> Summary Enhanced: This report, "The GenAI Divide: State of AI in Business 2025," presents preliminary findings from Project NANDA’s research into enterprise Generative AI (GenAI) adoption between January and June 2025. Employing a multi-method approach—including a systematic review of over 300 public AI initiatives, structured interviews with 52 organizations, and surveys from 153 senior leaders—the study investigates the disconnect between GenAI investment and realized business transformation.\n\nThe central thesis identifies a pronounced GenAI Divide: despite $30–40 billion in enterprise GenAI investment, 95% of organizations report no measurable return. Only 5% of integrated GenAI pilots deliver significant, sustained value, typically measured in millions of dollars. This divide is not attributable to model quality, regulatory constraints, or infrastructure, but rather to organizational approaches and the inability of most systems to learn, adapt, and integrate with existing workflows.\n\nKey findings include:\n\n- High Adoption, Low Transformation: Over 80% of organizations have piloted general-purpose tools such as ChatGPT and Copilot, and nearly 40% have deployed them. However, these tools primarily enhance individual productivity without impacting profit-and-loss (P&L) outcomes. Custom or enterprise-grade systems face high rejection rates: while 60% of organizations evaluate such tools, only 20% reach pilot stage and a mere 5% achieve production deployment.\n- Sectoral Disparities: Of nine major sectors analyzed, only Technology and Media & Telecom exhibit meaningful structural change attributable to GenAI. In other sectors—such as Healthcare, Financial Services, and Advanced Industries—GenAI adoption remains superficial, with little evidence of disrupted business models or significant shifts in user behavior.\n- Pilot-to-Production Chasm: The report documents a steep attrition rate from pilot to production, especially for task-specific GenAI solutions. While generic LLM chatbots show high implementation rates, their perceived business value is limited. Enterprises, despite leading in pilot volume, lag in scaling successful deployments compared to mid-market firms, which report faster transitions from pilot to production.\n- The Learning Gap: The predominant barrier to GenAI scale is the lack of learning and adaptability in current systems. Most tools do not retain feedback, adapt to context, or improve over time, resulting in user resistance and abandonment for mission-critical tasks. Even frequent users of consumer LLMs express skepticism toward enterprise tools that lack memory and contextual awareness.\n- Shadow AI Economy: A significant proportion of employees circumvent official channels by using personal AI tools (e.g., ChatGPT, Claude) for work tasks. While only 40% of companies have purchased official LLM subscriptions, over 90% of surveyed employees report regular use of personal AI tools at work. This shadow usage often delivers higher ROI than sanctioned initiatives and highlights the inadequacy of current enterprise solutions.\n- Investment Bias: GenAI budgets are disproportionately allocated to sales and marketing (approximately 50–70%), driven by the visibility and ease of measuring outcomes in these functions. Conversely, high-ROI opportunities in back-office automation remain underfunded due to challenges in quantifying indirect benefits and lower organizational visibility.\n\nThe report identifies several persistent myths, including the assumptions that GenAI is rapidly transforming business, that enterprises are slow adopters, and that internal tool development is most effective. In reality, internal builds fail at twice the rate of external partnerships, and transformation remains rare despite high adoption.\n\nLimitations of the study include reliance on self-reported data, varying sample sizes across sectors, and potential inconsistencies in the definition of successful implementation.\n\nImplications for AI practitioners are clear: realizing GenAI’s business value requires prioritizing systems that learn, adapt, and integrate with organizational processes. Successful adopters demand process-specific customization and evaluate tools based on business outcomes rather than technical benchmarks. Furthermore, analyzing shadow AI usage can inform procurement and deployment strategies. Addressing the learning gap—rather than focusing solely on model quality or regulatory barriers—is essential for bridging the GenAI Divide and achieving scalable, transformative impact.

*The summary generated with a temperature setting of 1 appears easier to read and has a better flow compared to the initial version. The revised text seems more verbose and consists of simpler sentences again. The improved text was not substantially better than the first one.*


In [105]:
report.ImprovedSummary
#ImprovedSummary

'This report, "The GenAI Divide: State of AI in Business 2025," presents preliminary findings from Project NANDA’s research into enterprise Generative AI (GenAI) adoption between January and June 2025. Employing a multi-method approach—including a systematic review of over 300 public AI initiatives, structured interviews with 52 organizations, and surveys from 153 senior leaders—the study investigates the disconnect between GenAI investment and realized business transformation.\n\nThe central thesis identifies a pronounced GenAI Divide: despite $30–40 billion in enterprise GenAI investment, 95% of organizations report no measurable return. Only 5% of integrated GenAI pilots deliver significant, sustained value, typically measured in millions of dollars. This divide is not attributable to model quality, regulatory constraints, or infrastructure, but rather to organizational approaches and the inability of most systems to learn, adapt, and integrate with existing workflows.\n\nKey findin

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
